<a href="https://colab.research.google.com/github/TharunPatel20/UST-GenAIEngineer-cohort-4/blob/main/RAG_Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-core langchain-google-genai faiss-cpu python-dotenv #faiss is the vectordb

In [2]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception as e:
    print("No Colab secret found. Add GEMINI_API_KEY in Colab Secrets, then run this cell again. Details:", e)
print("key found:", bool(os.environ.get("GEMINI_API_KEY")))

key found: True


In [3]:
!pip install pypdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_18998/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Part 1 — Data Understanding & Preprocessing**

In [5]:
loader=PyPDFLoader("/content/troubleshooting_runbook.pdf")

In [6]:
documents=loader.load()
report=""

for page in documents:
  report+=page.page_content+'\n'

print("Total number of pages",len(documents))
print(report[:520]) # b. Text quality issues

Total number of pages 7
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team  |  Version: 3.2  |  Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-engine and user-profi


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=700,  #each chunk will have 700 tokens
    chunk_overlap=70, #overlap of 70 tokens for each chunk created
)

In [8]:
chunk=text_splitter.split_documents(documents)
print(len(chunk)) #

30


**Part 2 — Embedding & Vector Database**

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [10]:
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [11]:
vector_store=FAISS.from_documents(
    documents=chunk,
    embedding=embedding
)

In [12]:
print(f"Vector store created with {vector_store.index.ntotal} vectors")
print(f"Embedding dimension: {vector_store.index.d}")


Vector store created with 30 vectors
Embedding dimension: 3072


In [13]:
vector_store.save_local("faiss_index")
print("Saved to faiss_index/")

Saved to faiss_index/


In [14]:
query = "How to troubleshoot GPU memory issues?"
results = vector_store.similarity_search_with_score(query, k=3)

In [15]:
print(f"\nQuery: {query}")
print(f"Top {len(results)} results:")


Query: How to troubleshoot GPU memory issues?
Top 3 results:


In [16]:
for i, (doc,score) in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}, Page: {doc.metadata.get('page', 'unknown')}")
    print(f"Content: {doc.page_content[:400]}...")
    print(f"Score: {score}")


--- Result 1 ---
Source: /content/troubleshooting_runbook.pdf, Page: 5
Content: Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 6
Infrastructure Errors
MEM-OOM - Container OOMKilled
Symptom: Pods restart repeatedly; kubectl describe shows 'Last State: Terminated, Reason: OOMKilled,
Exit Code: 137'. Common on search-service (large result sets) and reporting-engine (big in-memory
aggregations).
Likely causes: Memory limit set be...
Score: 0.6285208463668823

--- Result 2 ---
Source: /content/troubleshooting_runbook.pdf, Page: 5
Content: Confirm the limit: kubectl get pod -o jsonpath='{.spec.containers[*].resources}'. 3. Capture a heap dump
before the next restart. 4. Identify the request that precedes each kill.
Fix: Raise the limit by 25 to 50 percent as a stop-gap, paginate or stream large results, add max-range
validation on report requests, and set JVM -XX:MaxRAMPercentage=70.
Prevention: Load test with production-sized datas...
Scor

**Part 3 — Retrieval Pipeline**

In [17]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [18]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [19]:
test_queries = [
    "How to fix GPU out of memory error?",
    "Steps to debug distributed training failure?",
    "What causes CUDA initialization failure?",
    "How to resolve model convergence issues?"
]

In [20]:
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    for k in [3, 5, 7]:
        # Override k per query
        results = vector_store.similarity_search(query, k=k)
        print(f"\n--- k={k} ({len(results)} chunks) ---")
        for i, doc in enumerate(results, 1):
            page = doc.metadata.get('page', 'unknown')
            preview = doc.page_content[:100].replace('\n', ' ')
            print(f"  {i}. Page {page}: {preview}...")


QUERY: How to fix GPU out of memory error?

--- k=3 (3 chunks) ---
  1. Page 5: Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data Page 6 ...
  2. Page 5: Confirm the limit: kubectl get pod -o jsonpath='{.spec.containers[*].resources}'. 3. Capture a heap ...
  3. Page 4: the number of pods and exceeded the database max_connections. Diagnostics: 1. Compare active vs idle...

--- k=5 (5 chunks) ---
  1. Page 5: Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data Page 6 ...
  2. Page 5: Confirm the limit: kubectl get pod -o jsonpath='{.spec.containers[*].resources}'. 3. Capture a heap ...
  3. Page 4: the number of pods and exceeded the database max_connections. Diagnostics: 1. Compare active vs idle...
  4. Page 5: reporting-engine. Diagnostics: 1. df -h and du -sh /var/log/* to find the largest directories. 2. Ch...
  5. Page 3: changed default behaviour. Diagnostics: 1. Find the first occurrence timesta

In [21]:
print(f"\n{'='*60}")
print("MMR SEARCH (diverse results)")
print(f"{'='*60}")
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.5}
)
mmr_results = mmr_retriever.invoke(test_queries[0])
for i, doc in enumerate(mmr_results, 1):
    page = doc.metadata.get('page', 'unknown')
    print(f"  {i}. Page {page}: {doc.page_content[:100]}...")


MMR SEARCH (diverse results)
  1. Page 5: Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 6
...
  2. Page 4: PgBouncer in transaction mode for services with many replicas.
Prevention: Alert when pool utilisati...
  3. Page 3: timeout of 5 seconds or less. 4. Check HPA status.
Fix: Restore any missing index, set explicit clie...
  4. Page 5: reporting-engine.
Diagnostics: 1. df -h and du -sh /var/log/* to find the largest directories. 2. Ch...
  5. Page 3: changed default behaviour.
Diagnostics: 1. Find the first occurrence timestamp and match it to a dep...


**Part 4 — Answer Generation (RAG)**

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

In [23]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a platform engineering expert. Answer the user's question using ONLY the provided context.

Rules:
- Cite sources inline like [Page 5], [Page 3] using the page numbers from context
- If context is insufficient, say: "I don't have enough information in the runbook to answer this."
- Be concise and actionable — this is a troubleshooting runbook
- Do NOT use outside knowledge

Context:
{context}

Question: {question}

Answer:
""")

In [24]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        page = doc.metadata.get('page', 'unknown')
        formatted.append(f"[Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)


In [46]:

from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(
    #model='gemini-3.8-flash'
    model = "gemini-3.1-flash-lite"
)

In [47]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [48]:
test_query =["what are the database errors","summarize the types of errors in the runbook"]


In [53]:
for query in test_query:
  print(f"QUERY: {query}")
  answer = rag_chain.invoke(query)
  print(f"ANSWER: {answer}\n")
  print(f"-"*50,"\n")

QUERY: what are the database errors
ANSWER: The database errors identified in the runbook are:

*   **DB-1205 (Lock wait timeout exceeded):** Occurs when queries fail with 'Lock wait timeout exceeded'. Symptoms include latency spikes over 2 seconds and failing writes, often caused by long-running transactions, missing indexes, or batch updates [Page 5].
*   **DB-POOL (Connection pool exhausted):** Occurs when services experience timeouts waiting for a connection or lack available connection slots. Often caused by connection leaks, insufficient pool sizing, or slow queries [Page 5].

-------------------------------------------------- 

QUERY: summarize the types of errors in the runbook
ANSWER: The runbook covers the following error codes:

*   **ERR-401 (Expired/invalid authentication token):** Characterized by token expiration issues, clock skew, or JWKS cache failures [Page 2, Page 7].
*   **ERR-429 (Rate limit exceeded):** Involves high traffic from specific API keys or retry storms